# t-Tests: the Three Tests, Rerun Without $\sigma$

Same ten heights, same three hypothesis sets — but this time $\sigma$ is honestly unknown, so every computation moves to the **t-statistic**, and every curve is drawn around 0 instead of 66.7. The right-tailed test now reads $t_{obs} = 1.77$ against the $t_9$ curve: $p = 0.0552 > 0.05$ — **the exact opposite verdict** from the known-$\sigma$ z-test, because the uncertainty of estimating $\sigma$ made the same evidence insufficient. The two-tailed p doubles to $0.1103$ (still no rejection), and the decreased-heights sample ($\bar{x} = 64.252$) gives $t_{obs} = -2.487$, $p = 0.0173$ — a rejection that survives even without $\sigma$. It is based on the ninth video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

Notebook 04 ran the three tests with a borrowed $\sigma$; notebook 08 built the honest statistic. This notebook puts them together — the complete t-test, three tails, one dataset — and stares at the most instructive outcome in the lesson: a rejection that evaporates when the assumption of known $\sigma$ is given up.

After completing this notebook you will be able to:

- Rerun right-, two-, and left-tailed tests with the t-statistic, reading p-values off the $t_{n-1}$ curve centered at 0.
- Reproduce the video's numbers: $p = 0.0552$, $0.1103$, and $0.0173$.
- Explain the verdict flip against the known-$\sigma$ z-test — and why it is a feature, not a bug.
- Handle the two-tailed absolute value correctly for negative observed statistics.
- Quantify what it would have taken (in $s$, or in $n$) for the right-tailed rejection to survive.

## 2. Theory

### The setup, minus one assumption

The sample: ten 18-year-olds, $\bar{x} = 68.442$ — and now also $s = 3.113$, because with $\sigma$ unknown the sample must supply its own scale. The three hypothesis sets are unchanged ($H_0: \mu = 66.7$ against $>$, $\ne$, $<$). One practical drawback: all computations happen **in terms of the t-statistic rather than the sample mean directly** — instead of plotting distributions around 66.7, everything is drawn around **0**. Under $H_0$ the statistic

$$
T = \frac{\bar{X} - 66.7}{S/\sqrt{10}} \;\sim\; t_9
$$

### The right-tailed test

$$
t_{obs} = \frac{68.442 - 66.7}{3.113/\sqrt{10}} = 1.77, \qquad
p = P(T > 1.77 \mid H_0) = 0.0552
$$

Since $0.0552 > 0.05$: **do not reject $H_0$.** This is the exact opposite of the known-$\sigma$ result ($p = 0.0332$, reject). Nothing about the sample changed — what changed is that the extra uncertainty from estimating the population's spread is now priced in, and *suddenly the evidence you had is not enough*.

### The two-tailed test

"More extreme" now counts both directions, so the p-value is the probability that $|T|$ exceeds the observed value. (Our observed statistic is positive, so using it directly works; in general you take the **absolute value**, in case $t_{obs}$ is negative.) By symmetry the left tail contributes the same area:

$$
p = P(|T| > 1.77) = 2 \times 0.0552 = 0.1103 > 0.05 \quad\Rightarrow\quad \text{do not reject.}
$$

### The left-tailed test

For the alternative "heights decreased," use again the sample with $\bar{x} = 64.252$ (imagining the same $s$):

$$
t_{obs} = \frac{64.252 - 66.7}{3.113/\sqrt{10}} = -2.487, \qquad
p = P(T < -2.487) = 0.0173 < 0.05 \quad\Rightarrow\quad \text{reject } H_0:
$$

accept that the population mean has decreased. This rejection *survives* the loss of $\sigma$ — its evidence was strong enough to pay the fat-tail surcharge (though note it no longer clears $\alpha = 0.01$, as it did in the z-world where $p$ was 0.0049).

### The scoreboard

| test | z-world (σ = 3 known) | t-world (s = 3.113 estimated) |
|---|---|---|
| right-tailed | $p = 0.0332$ → reject | $p = 0.0552$ → **don't reject** |
| two-tailed | $p = 0.0663$ → don't reject | $p = 0.1103$ → don't reject |
| left-tailed ($\bar{x} = 64.252$) | $p = 0.0049$ → reject (even at 0.01) | $p = 0.0173$ → reject (only at 0.05) |

Every p-value grew — honesty about $\sigma$ always costs evidence — and one verdict flipped outright.

## 3. Worked Examples

### Example 1: right-tailed arithmetic

$s/\sqrt{n} = 3.113/\sqrt{10} = 0.984$; $t_{obs} = 1.742/0.984 = 1.77$; look up the $t_9$ right tail beyond 1.77: 0.0552. Compare with the critical-value view: $t_{0.95,9} = 1.833 > 1.77$ — the observation falls short of the bar, same conclusion.

### Example 2: the two-tailed doubling, carefully

$p_{two} = P(|T| > |t_{obs}|) = P(T > 1.77) + P(T < -1.77) = 2 \times 0.0552 = 0.1103$. The absolute value matters: had the observed statistic been $-1.77$, the *same* two-tailed p results — the two-tailed test is indifferent to direction, by design.

### Example 3: the left-tailed test against its z-twin

t-world: $t_{obs} = -2.487$, $p = 0.0173$. z-world (σ = 3): $z_{obs} = -2.580$, $p = 0.0049$ — about 3.5× smaller. Both reject at $\alpha = 0.05$, but only the z-version rejects at $0.01$: the fat tails don't always flip verdicts, yet they always tax the p-value.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project) and the t-machinery from the previous notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from math import gamma, sqrt, pi

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(99)

def t_pdf(t, df):
    """Density of the Student t-distribution with df degrees of freedom."""
    coeff = gamma((df + 1) / 2) / (sqrt(df * pi) * gamma(df / 2))
    return coeff * (1 + np.asarray(t, dtype=float) ** 2 / df) ** (-(df + 1) / 2)

def t_tail(t_value, df):
    """P(T > t_value) for T ~ t_df, by numeric integration."""
    grid = np.linspace(0, abs(t_value), 20_000)
    cdf_half = np.trapezoid(t_pdf(grid, df), grid)
    tail = 0.5 - cdf_half
    return tail if t_value >= 0 else 1 - tail

Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))

def t_test(x_bar, s, n, mu0, tail, alpha=0.05, label=""):
    """The complete t-test from summary statistics."""
    t_obs = (x_bar - mu0) / (s / np.sqrt(n))
    df = n - 1
    p = {"right": t_tail(t_obs, df),
         "left": 1 - t_tail(t_obs, df),
         "two": 2 * t_tail(abs(t_obs), df)}[tail]
    verdict = "REJECT H0" if p < alpha else "fail to reject H0"
    print(f"{label}: t_obs = {t_obs:.3f} (df = {df}), p = {p:.4f} -> {verdict}")
    return t_obs, p

### The video's three tests

In [ ]:
heights = np.array([64.12, 71.15, 66.41, 70.17, 73.13,
                    65.10, 69.16, 67.39, 71.97, 65.82])
MU0 = 66.7
x_bar, s, n = heights.mean(), heights.std(ddof=1), len(heights)
print(f"sample: x-bar = {x_bar:.3f}, s = {s:.3f}, n = {n}\n")

t_r, p_r = t_test(x_bar, s, n, MU0, "right", label="right-tailed (mu > 66.7)")
t_2, p_2 = t_test(x_bar, s, n, MU0, "two", label="two-tailed   (mu != 66.7)")
t_l, p_l = t_test(64.252, s, n, MU0, "left", label="left-tailed  (mu < 66.7, x-bar = 64.252)")

print(f"\ntwo-tailed p is exactly double the right-tailed: {p_2 / p_r:.3f}x")
print(f"left-tailed p also below alpha = 0.01? {p_l < 0.01}")

### The scoreboard: t-world vs z-world

The same three tests with the borrowed $\sigma = 3$ (notebook 04) against today's honest versions.

In [ ]:
SE_Z = 3.0 / np.sqrt(10)
z_r = (x_bar - MU0) / SE_Z
z_l = (64.252 - MU0) / SE_Z
rows = [
    ("right-tailed", 1 - Phi(z_r), p_r),
    ("two-tailed", 2 * (1 - Phi(abs(z_r))), p_2),
    ("left-tailed", Phi(z_l), p_l),
]
print(f"{'test':>14s} {'z-world p':>10s} {'t-world p':>10s}   verdicts (alpha = 0.05)")
for name, pz, pt in rows:
    vz = "reject" if pz < 0.05 else "keep"
    vt = "reject" if pt < 0.05 else "keep"
    flag = "  <-- FLIP" if vz != vt else ""
    print(f"{name:>14s} {pz:>10.4f} {pt:>10.4f}   {vz:>6s} / {vt}{flag}")
print("\nEvery p grew -- estimating sigma always costs evidence -- and the")
print("right-tailed rejection did not survive the honesty.")

## 5. Visualizations

### The three tests on the $t_9$ curve

Everything now lives around 0: the shaded areas are the video's three p-values.

In [ ]:
t_axis = np.linspace(-4.5, 4.5, 700)
density = t_pdf(t_axis, 9)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
panels = [
    (f"right-tailed\n$t$ = {t_r:.2f}", t_axis >= t_r, [t_r], p_r, "don't reject"),
    (f"two-tailed\n$|t|$ = {t_r:.2f}", np.abs(t_axis) >= abs(t_2), [-t_2, t_2],
     p_2, "don't reject"),
    (f"left-tailed\n$t$ = {t_l:.2f}", t_axis <= t_l, [t_l], p_l, "REJECT"),
]
for ax, (title, region, marks, p, verdict) in zip(axes, panels):
    ax.plot(t_axis, density, color="black", linewidth=1.6)
    ax.fill_between(t_axis, density, where=region, color="tab:red", alpha=0.55)
    for m in marks:
        ax.axvline(m, color="tab:red", linewidth=1.6)
    ax.axvline(0, color="black", linestyle=":", linewidth=1)
    ax.set_title(f"{title}\np = {p:.4f} -> {verdict}", fontsize=10)
    ax.set_xlabel("t")
axes[0].set_ylabel("density under $H_0$: $t_9$")
plt.suptitle("The three t-tests: same logic as notebook 04, new statistic, new curve ($\\alpha$ = 0.05)")
plt.tight_layout()
plt.show()

### The flip, anatomized

The right-tailed test in both worlds. The statistics barely differ (1.84 vs 1.77); the tails do — and the verdict with them.

In [ ]:
z_axis = t_axis
fig, ax = plt.subplots(figsize=(10.5, 4.8))
ax.plot(z_axis, np.exp(-0.5 * z_axis**2) / np.sqrt(2 * np.pi), color="tab:green",
        linewidth=1.8, label="z-world: N(0,1), z = 1.836, p = 0.0332")
ax.plot(t_axis, density, color="tab:red", linewidth=1.8,
        label=f"t-world: $t_9$, t = {t_r:.3f}, p = {p_r:.4f}")
ax.fill_between(z_axis, np.exp(-0.5 * z_axis**2) / np.sqrt(2 * np.pi),
                where=z_axis >= z_r, color="tab:green", alpha=0.35)
ax.fill_between(t_axis, density, where=t_axis >= t_r, color="tab:red", alpha=0.35)
ax.axvline(z_r, color="tab:green", linestyle="--", linewidth=1.2)
ax.axvline(t_r, color="tab:red", linestyle="--", linewidth=1.2)
ax.set_xlim(-4, 4.5)
ax.set_xlabel("standardized statistic")
ax.set_ylabel("density under $H_0$")
ax.set_title("Nearly the same statistic, fatter tail, opposite verdict")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Rerun the right-tailed t-test at $\alpha = 0.10$. What is the verdict now, and what does the pair of outcomes (0.05 vs 0.10) tell you about reporting p-values rather than bare verdicts?

**Exercise 2.** A colleague observes $t_{obs} = -1.77$ in a two-tailed test and computes $p = P(T > -1.77) = 0.945$. Diagnose the error and give the correct p-value.

### Intermediate

**Exercise 3.** *How tight would the sample have to be?* With $\bar{x} = 68.442$ and $n = 10$ fixed, find the largest sample standard deviation $s^*$ for which the right-tailed t-test still rejects at $\alpha = 0.05$ ($t_{0.95,9} = 1.833$). Compare $s^*$ with the assumed $\sigma = 3$ of the z-world and comment.

**Exercise 4.** *Or how much more data?* If the same $\bar{x} = 68.442$ and $s = 3.113$ had come from $n = 30$ people, compute $t_{obs}$, the degrees of freedom, and the p-value. Which two distinct effects improved it?

### Advanced

**Exercise 5.** *The price of honesty, in power.* By simulation, estimate the power of the right-tailed z-test ($\sigma = 3$ known, threshold $z > 1.645$) and of the t-test (threshold $t > t_{0.95,9}$) when the true population is $N(68.4, 3^2)$, $n = 10$. How many points of power does estimating $\sigma$ cost here?

**Exercise 6.** *A full t-test from raw data.* Notebook 07 tested the delivery times `[2.4, 3.1, 2.2, 2.9, 2.5, 2.8, 2.1, 2.7, 3.0, 2.3, 2.6, 2.2]` against $H_0: \mu = 3.0$ (left-tailed) using a conveniently "known" $\sigma = 0.8$. Redo it honestly: compute $s$ from the data, run the t-test, and compare both the statistic and the verdict with the z-version. Why did honesty *help* this time?

## 7. Solutions

In [ ]:
# Exercise 1: alpha = 0.10
_ = t_test(x_bar, s, n, MU0, "right", alpha=0.10,
           label="right-tailed at alpha = 0.10")
print("\nThe same evidence rejects at 0.10 and fails at 0.05 -- a bare verdict")
print("hides how close the call was. Reporting p = 0.0552 lets every reader")
print("apply their own alpha.")

In [ ]:
# Exercise 2: the two-tailed absolute value
p_colleague = t_tail(-1.77, 9)                 # P(T > -1.77): nearly everything
p_correct = 2 * t_tail(abs(-1.77), 9)
print(f"the colleague's P(T > -1.77) = {p_colleague:.4f} -- nearly the whole curve,")
print("because almost all of t_9 sits above -1.77. That's not extremeness.")
print(f"correct two-tailed p = 2 * P(T > |t_obs|) = {p_correct:.4f}")
print("\nThe error: forgetting the absolute value. 'As extreme as' a negative")
print("statistic means further from 0 in BOTH directions, not larger than it.")

In [ ]:
# Exercise 3: the largest s that still rejects
t_crit = 1.8331
s_star = (x_bar - MU0) * np.sqrt(n) / t_crit
print(f"reject requires t_obs >= {t_crit} -> s <= 1.742 * sqrt(10)/{t_crit} "
      f"= {s_star:.3f}")
_ = t_test(x_bar, 2.99, n, MU0, "right", label="check just inside  (s = 2.99)")
_ = t_test(x_bar, 3.02, n, MU0, "right", label="check just outside (s = 3.02)")
print(f"\ns* = {s_star:.2f} is essentially the sigma = 3 the z-world assumed:")
print("the borrowed sigma sat right at the edge of what the data could afford --")
print("which is exactly why pricing in its uncertainty tipped the verdict.")

In [ ]:
# Exercise 4: same summary stats, n = 30
_ = t_test(x_bar, s, 30, MU0, "right", label="right-tailed with n = 30")
print("\nTwo distinct effects: the standard error shrinks (t_obs grows from 1.77")
print("to 3.07), AND the curve slims (t_29 is much closer to normal than t_9).")
print("Both push the p-value down -- data attacks the problem from both ends.")

In [ ]:
# Exercise 5: power cost of estimating sigma
trials = 100_000
true_samples = rng.normal(68.4, 3.0, size=(trials, 10))
xbars = true_samples.mean(axis=1)
svals = true_samples.std(axis=1, ddof=1)

z_stats = (xbars - MU0) / (3.0 / np.sqrt(10))
t_stats = (xbars - MU0) / (svals / np.sqrt(10))
z95 = float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(0.95) - 1))

power_z = (z_stats > z95).mean()
power_t = (t_stats > 1.8331).mean()
print(f"power of the z-test (sigma known):     {power_z:.1%}")
print(f"power of the t-test (sigma estimated): {power_t:.1%}")
print(f"\nEstimating sigma costs about {power_z - power_t:.1%} of power at n = 10 --")
print("the price of honesty, payable in either extra risk or extra data.")

In [ ]:
# Exercise 6: honest delivery-time test
deliveries = np.array([2.4, 3.1, 2.2, 2.9, 2.5, 2.8, 2.1, 2.7, 3.0, 2.3, 2.6, 2.2])
d_mean, d_s, d_n = deliveries.mean(), deliveries.std(ddof=1), len(deliveries)
print(f"x-bar = {d_mean:.3f}, s = {d_s:.3f} (vs the 'known' 0.8), n = {d_n}")
_ = t_test(d_mean, d_s, d_n, 3.0, "left", label="honest left-tailed t-test")

z_del = (d_mean - 3.0) / (0.8 / np.sqrt(d_n))
print(f"z-version had z = {z_del:.3f}, p = {Phi(z_del):.4f}")
print("\nHonesty helped because the data's real spread (s = 0.34) is far SMALLER")
print("than the assumed 0.8: the borrowed sigma was overly pessimistic, and the")
print("honest t-test sees the drop in delivery times far more sharply.")

## 8. Conclusion

- With $\sigma$ unknown, all three tests run on the **t-statistic**, with curves drawn around 0 and p-values read off $t_{n-1}$ ($t_9$ here).
- The video's scoreboard: right-tailed $p = 0.0552$ → **don't reject** (the z-world rejected — the flip is the uncertainty of estimating $\sigma$, priced in); two-tailed $p = 0.1103$ (double, with an absolute value in general); left-tailed ($\bar{x} = 64.252$) $p = 0.0173$ → reject, though no longer at $0.01$.
- Every honest p-value exceeds its z-twin — sometimes flipping the verdict, always taxing the evidence: about 9 points of power at $n = 10$ in the exercise's measurement.
- The escape routes, quantified: the rejection would have survived with $s \le 3.00$ (the z-world's assumed $\sigma$ sat exactly on that edge), or with $n = 30$ (where both the statistic and the curve improve).
- And honesty can cut the other way: when the borrowed $\sigma$ was pessimistic (delivery times, $s = 0.34$ vs assumed $0.8$), the t-test sees the effect *more* sharply.

Next: the lesson's destination — comparing two groups at once. Two-sample tests and A/B testing.